# Tutorial 06c — Fused Attention, **Minimal**: Triton on CUDA, Ampere/Ada only

> A companion to **`tutorials_jupyter/06-fused-attention.ipynb`** (OpenAI's Triton
> **FlashAttention-2**). **06a** unpacked the forward pass; **06b** unpacked the backward.
> This notebook asks a different question:
>
> ### Of that 760-line cell, how much is actually running on *your* GPU?
>
> The tutorial is a **portable** kernel. It must work on AMD (HIP) and NVIDIA; on Ampere, Ada,
> Hopper and Blackwell; in fp16 and fp8. Portability is paid for in `if` statements — and you
> still have to read every one of them. Here we delete every branch that a **CUDA Ampere/Ada**
> card cannot take, and then measure, honestly, what that cost us.
>
> **Persona:** entry-to-intermediate practitioner (you know PyTorch, the GPU memory hierarchy,
> and roughly what a Triton kernel is). Same voice as **06a**/**06b** and course Chapters 16a–16d.

### Prerequisite

**06a**, at least its §2 (online softmax) and §5 (the causal `STAGE` split). You should already
know that `_attn_fwd` loops over key blocks, keeps a running max `m_i` and sum `l_i`, and writes
`O` plus a per-row logsumexp `M`. We are not re-deriving the algorithm — we are deleting
scaffolding from around it.

### Which GPUs is this for?

| Architecture | Compute capability | Example cards | This notebook |
|---|---|---|---|
| Ampere | 8.0, 8.6 | A100, A10, RTX 30xx | **target** |
| Ada Lovelace | 8.9 | L40S, RTX 40xx | **target** |
| Hopper | 9.0 | H100, H200 | keep the tutorial's TMA path |
| Blackwell | 10.x | B200, RTX 50xx | keep TMA + warp specialization |
| AMD (HIP) | — | MI250, MI300 | keep the HIP branch |


### How to use this notebook

Unlike 06a/06b, **the kernels here really run.** If you have a CUDA GPU and `triton` + `torch`
installed, every code cell executes: it compiles the stripped kernel, checks it against a PyTorch
reference, disassembles it to PTX, and benchmarks it.

If you *don't* have that, nothing breaks. The kernel cells still **define** (via a small no-op shim
in §1), the exercises still run — they are pure NumPy/Python — and the recorded numbers are in the
figures. All figures were measured on an **RTX 4080 SUPER (sm_89, Ada), Triton 3.7.0, torch 2.12.0**.

### Learning objectives

By the end you will be able to:

- Name the **five arch-gate predicates** in the tutorial and say which code each one guards.
- Explain why a **TMA tensor descriptor** costs nothing on Ampere/Ada — and prove it by reading PTX.
- Explain why `warp_specialize=True` is **inert** on anything below Blackwell.
- Decide, per deletion, whether it is **free** (most) or whether it **costs real throughput** (FP8).
- Write a ~120-line fused-attention forward that matches the 236-line original within a few percent.
- Say precisely **what you gave up**, and on which GPU you would have to put it back.


## 0. A map: the five gates

Before a single kernel line, the tutorial defines five predicates. Everything arch-specific hangs
off them. Keep this table open — each row is a section below.

| Predicate | Returns true on | What it guards | Deleted in |
|---|---|---|---|
| `is_hip()` | AMD GPUs | `waves_per_eu`, `allow_flush_denorm`, `NUM_STAGES_OPTIONS=[1]` | §2 |
| `is_cuda()` | NVIDIA GPUs | (kept — this *is* our target) | — |
| `supports_host_descriptor()` | capability ≥ 9.0 | building `TensorDescriptor`s on the **host** | §3 |
| `is_hopper()` | capability == 9.0 | `keep()` config pruning, the `IS_HOPPER` acc path | §4, §6 |
| `is_blackwell()` | capability == 10.0 | `maxnreg`, real **warp specialization** | §4 |

The flow below is what happens when the tutorial's `_attention.forward` runs. Trace the **green**
path — that is the only one an Ampere/Ada card can take.

```mermaid
graph TD
  A["_attention.forward(q,k,v)"] --> B{"is_hip()?"}
  B -- yes --> C["waves_per_eu, allow_flush_denorm"]
  B -- no --> D{"supports_host_descriptor()<br/>cap >= 9.0 ?"}
  D -- yes --> E["host TensorDescriptor(q,k,v,o)<br/>TMA hardware path"]
  D -- no --> F["pass raw tensors<br/>kernel builds device descriptors"]
  F --> G{"is_blackwell() and warp_specialize?"}
  G -- yes --> H["maxnreg=80/168<br/>real warp specialization"]
  G -- no --> I["plain tl.range loop"]
  I --> J["_attn_fwd on sm_80 / 86 / 89"]
  style F fill:#c6f6d5,stroke:#2f855a
  style I fill:#c6f6d5,stroke:#2f855a
  style J fill:#c6f6d5,stroke:#2f855a
```


### Simulate the gates before you trust them

The predicates are one-liners over `torch.cuda.get_device_capability()`. We can reproduce them in
plain Python and see which branches each architecture takes — **no GPU required**. Run this and
read the `Ada` row: everything is `False` except `is_cuda`.


In [ ]:
def gates(backend: str, major: int) -> dict[str, bool]:
    """Reproduce the tutorial's five predicates from (backend, capability major)."""
    is_cuda = backend == "cuda"
    return {
        "is_hip": backend == "hip",
        "is_cuda": is_cuda,
        "supports_host_descriptor": is_cuda and major >= 9,
        "is_hopper": is_cuda and major == 9,
        "is_blackwell": is_cuda and major == 10,
    }


ARCHS = [("MI300 (HIP)", "hip", 9), ("Ampere A100", "cuda", 8), ("Ada RTX 4080", "cuda", 8),
         ("Hopper H100", "cuda", 9), ("Blackwell B200", "cuda", 10)]

names = list(gates("cuda", 8))
print(f"{'GPU':<16}" + "".join(f"{n:>26}" for n in names))
print("-" * (16 + 26 * len(names)))
for label, backend, major in ARCHS:
    g = gates(backend, major)
    print(f"{label:<16}" + "".join(f"{str(g[n]):>26}" for n in names))

ada = gates("cuda", 8)
assert ada["is_cuda"] and not any(ada[k] for k in ada if k != "is_cuda"), "Ada should trip only is_cuda"
print("\nOn Ampere/Ada exactly ONE gate is True: is_cuda. Everything else is dead code.")


## 1. Ask your own GPU

Now the real thing. This cell imports `torch`/`triton` if they exist and sets `HAS_TRITON`.

If they don't exist, it installs a **no-op shim** so that every `@triton.jit` kernel below still
*defines* without error (the decorator becomes the identity function). Nothing is executed on a
GPU, but the notebook stays readable and the exercises still run. This is also what lets
`course/review-notebook.py` execute this notebook under plain Python.


In [ ]:
try:
    import torch
    import triton
    import triton.language as tl

    HAS_TRITON = torch.cuda.is_available()
except ImportError:
    HAS_TRITON = False

    class _Shim:
        """No-op stand-in: @triton.jit -> identity, triton.Config(...) -> harmless object."""

        def __getattr__(self, _name):
            def _deco(*args, **kwargs):
                if len(args) == 1 and not kwargs and callable(args[0]):
                    return args[0]
                return lambda f: f
            return _deco

    triton = tl = _Shim()

if HAS_TRITON:
    major, minor = torch.cuda.get_device_capability()
    print(f"device      : {torch.cuda.get_device_name()}")
    print(f"capability  : {major}.{minor}")
    print(f"triton      : {triton.__version__}")
    print(f"torch       : {torch.__version__}")
    print()
    g = gates("cuda", major)
    for k, v in g.items():
        print(f"  {k:<26} {v}")
    if major == 8:
        print("\n-> Ampere/Ada. Every deletion in this notebook is one you can make.")
    else:
        print(f"\n-> capability {major}.x: you are NOT the target; read on, but keep your arch's paths.")
else:
    print("No torch+CUDA here. Kernels will DEFINE (via the shim) but not RUN.")
    print("Exercises are pure NumPy and run fine. Figures hold the recorded numbers.")


## 2. Deletion 1 — the HIP branch

The smallest one. AMD's compiler wants two knobs NVIDIA's does not, and prefers a single
pipeline stage:

```python
if is_hip():
    NUM_STAGES_OPTIONS = [1]
else:
    NUM_STAGES_OPTIONS = [2, 3, 4]
...
if is_hip():
    waves_per_eu = 3 if HEAD_DIM_K <= 64 else 2
    extra_kern_args = {"waves_per_eu": waves_per_eu, "allow_flush_denorm": True}
```

`waves_per_eu` is an AMD occupancy hint; `allow_flush_denorm` lets CDNA flush denormals to zero.
Neither exists in the CUDA backend, and `extra_kern_args` is `{}` on NVIDIA.

**Cost of deleting: zero.** On a CUDA card these lines only ever evaluate a `False` and move on.
This is the easy kind of dead code — obviously dead, and obviously safe to remove. The rest of
the chapter is about the *non*-obvious kind.


## 3. Deletion 2 — TMA tensor descriptors

This is the big one, and the one people get wrong.

The **Tensor Memory Accelerator** is a hardware unit that copies multi-dimensional tiles between
global and shared memory without occupying the threads. One thread hands it a *descriptor* and
the hardware streams the tile. It is introduced in **compute capability 9.0 (Hopper)** and later
([CUDA Programming Guide — Asynchronous Data
Copies](https://docs.nvidia.com/cuda/cuda-programming-guide/04-special-topics/async-copies.html)).
In PTX it appears as `cp.async.bulk.tensor`. **Ampere and Ada have no such unit.**

The tutorial therefore has *two* descriptor paths:

- **Host-side** — `TensorDescriptor(q, shape=..., strides=..., block_shape=...)` built in Python,
  plus a `_host_descriptor_pre_hook` that patches each descriptor's `block_shape` every time the
  autotuner picks a new `BLOCK_M`/`BLOCK_N`. Guarded by `supports_host_descriptor()`, i.e. **≥ 9.0**.
- **Device-side** — `tl.make_tensor_descriptor(ptr, shape, strides, block_shape)` built *inside*
  the kernel by `_maybe_make_tensor_desc`. This runs on **everything**, including your Ada card.


So on Ampere/Ada, `supports_host_descriptor()` is `False`, raw tensors are passed to the kernel,
and `_maybe_make_tensor_desc` builds a **device-side** descriptor for each of `q, k, v, o`:

```python
@triton.jit
def _maybe_make_tensor_desc(desc_or_ptr, shape, strides, block_shape):
    if isinstance(desc_or_ptr, tl.tensor_descriptor):
        return desc_or_ptr
    else:
        return tl.make_tensor_descriptor(desc_or_ptr, shape, strides, block_shape)
```

Which raises the only question that matters:

> **On a GPU with no TMA hardware, what does a tensor descriptor actually compile to?**

The docs say *"On NVIDIA GPUs with TMA support, this will result in a TMA descriptor object and
loads and stores from the descriptor will be backed by the TMA hardware"*
([`tl.make_tensor_descriptor`](https://triton-lang.org/main/python-api/generated/triton.language.make_tensor_descriptor.html))
— and say nothing about GPUs without it. Let's not guess. Let's disassemble.


### The probe: one kernel, three ways to load

Below are three **probe kernels** — deliberately tiny, non-causal, fp16, no autotuning. They
compute the same fused attention with the same math, and differ *only* in how they address memory:

1. `_probe_desc` — device tensor descriptors (what the tutorial does on your card).
2. `_probe_ptr` — plain pointer arithmetic (what the tutorial's **backward** already does).
3. `_probe_bp` — `tl.make_block_ptr` / `tl.advance` (the older Triton idiom).

All three are pinned to `BLOCK_M=128, BLOCK_N=64, HEAD_DIM=64, num_stages=2, num_warps=4` — the
exact single config the tutorial itself pins under `PYTEST_VERSION` "for reproducibility". Same
config, same math, three sources. If the descriptor buys anything on Ada, the PTX will show it.


In [ ]:
BM, BN, HD, STAGES, WARPS = 128, 64, 64, 2, 4


@triton.jit
def _probe_ptr(Q, K, V, sm_scale, M, Out, sqz, sqh, sqm, sqk, Z, H,
               N_CTX: tl.constexpr, HEAD_DIM: tl.constexpr,
               BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    start_m, off_hz = tl.program_id(0), tl.program_id(1)
    base = (off_hz // H).to(tl.int64) * sqz + (off_hz % H).to(tl.int64) * sqh
    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n, offs_d = tl.arange(0, BLOCK_N), tl.arange(0, HEAD_DIM)
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32) + 1.0
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)
    qk_scale = sm_scale * 1.44269504
    q = tl.load(Q + base + offs_m[:, None] * sqm + offs_d[None, :] * sqk)
    K_ptr = K + base + offs_n[None, :] * sqm + offs_d[:, None] * sqk   # kT tile
    V_ptr = V + base + offs_n[:, None] * sqm + offs_d[None, :] * sqk
    for _ in tl.range(0, N_CTX, BLOCK_N):
        qk = tl.dot(q, tl.load(K_ptr))
        m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
        p = tl.math.exp2(qk * qk_scale - m_ij[:, None])
        alpha = tl.math.exp2(m_i - m_ij)
        acc = acc * alpha[:, None]
        acc = tl.dot(p.to(tl.float16), tl.load(V_ptr), acc)
        l_i = l_i * alpha + tl.sum(p, 1)
        m_i = m_ij
        K_ptr += BLOCK_N * sqm
        V_ptr += BLOCK_N * sqm
    m_i += tl.math.log2(l_i)
    acc = acc / l_i[:, None]
    tl.store(M + off_hz * N_CTX + offs_m, m_i)
    tl.store(Out + base + offs_m[:, None] * sqm + offs_d[None, :] * sqk, acc.to(tl.float16))


@triton.jit
def _probe_bp(Q, K, V, sm_scale, M, Out, sqz, sqh, sqm, sqk, Z, H,
              N_CTX: tl.constexpr, HEAD_DIM: tl.constexpr,
              BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    start_m, off_hz = tl.program_id(0), tl.program_id(1)
    base = (off_hz // H).to(tl.int64) * sqz + (off_hz % H).to(tl.int64) * sqh
    Qb = tl.make_block_ptr(Q + base, (N_CTX, HEAD_DIM), (sqm, sqk), (start_m * BLOCK_M, 0), (BLOCK_M, HEAD_DIM), (1, 0))
    Kb = tl.make_block_ptr(K + base, (HEAD_DIM, N_CTX), (sqk, sqm), (0, 0), (HEAD_DIM, BLOCK_N), (0, 1))
    Vb = tl.make_block_ptr(V + base, (N_CTX, HEAD_DIM), (sqm, sqk), (0, 0), (BLOCK_N, HEAD_DIM), (1, 0))
    Ob = tl.make_block_ptr(Out + base, (N_CTX, HEAD_DIM), (sqm, sqk), (start_m * BLOCK_M, 0), (BLOCK_M, HEAD_DIM), (1, 0))
    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32) + 1.0
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)
    qk_scale = sm_scale * 1.44269504
    q = tl.load(Qb)
    for _ in tl.range(0, N_CTX, BLOCK_N):
        qk = tl.dot(q, tl.load(Kb))
        m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
        p = tl.math.exp2(qk * qk_scale - m_ij[:, None])
        alpha = tl.math.exp2(m_i - m_ij)
        acc = acc * alpha[:, None]
        acc = tl.dot(p.to(tl.float16), tl.load(Vb), acc)
        l_i = l_i * alpha + tl.sum(p, 1)
        m_i = m_ij
        Kb, Vb = tl.advance(Kb, (0, BLOCK_N)), tl.advance(Vb, (BLOCK_N, 0))
    m_i += tl.math.log2(l_i)
    acc = acc / l_i[:, None]
    tl.store(M + off_hz * N_CTX + offs_m, m_i)
    tl.store(Ob, acc.to(tl.float16))


@triton.jit
def _probe_desc(sm_scale, M, Z, H, desc_q, desc_k, desc_v, desc_o,
                N_CTX: tl.constexpr, HEAD_DIM: tl.constexpr,
                BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    start_m, off_hz = tl.program_id(0), tl.program_id(1)
    off_z, off_h = off_hz // H, off_hz % H
    y_dim = Z * H * N_CTX
    dq = tl.make_tensor_descriptor(desc_q, [y_dim, HEAD_DIM], [HEAD_DIM, 1], [BLOCK_M, HEAD_DIM])
    dk = tl.make_tensor_descriptor(desc_k, [y_dim, HEAD_DIM], [HEAD_DIM, 1], [BLOCK_N, HEAD_DIM])
    dv = tl.make_tensor_descriptor(desc_v, [y_dim, HEAD_DIM], [HEAD_DIM, 1], [BLOCK_N, HEAD_DIM])
    do = tl.make_tensor_descriptor(desc_o, [y_dim, HEAD_DIM], [HEAD_DIM, 1], [BLOCK_M, HEAD_DIM])
    offset_y = off_z * (N_CTX * H) + off_h * N_CTX
    qo = offset_y + start_m * BLOCK_M
    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32) + 1.0
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)
    qk_scale = sm_scale * 1.44269504
    q = dq.load([qo, 0])
    off = offset_y
    for _ in tl.range(0, N_CTX, BLOCK_N):
        qk = tl.dot(q, dk.load([off, 0]).T)
        m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
        p = tl.math.exp2(qk * qk_scale - m_ij[:, None])
        alpha = tl.math.exp2(m_i - m_ij)
        acc = acc * alpha[:, None]
        acc = tl.dot(p.to(tl.float16), dv.load([off, 0]), acc)
        l_i = l_i * alpha + tl.sum(p, 1)
        m_i = m_ij
        off += BLOCK_N
    m_i += tl.math.log2(l_i)
    acc = acc / l_i[:, None]
    tl.store(M + off_hz * N_CTX + offs_m, m_i)
    do.store([qo, 0], acc.to(tl.float16))

print("three probe kernels defined (descriptors / plain pointers / block pointers)")


### Compile all three, then count instructions

`tl.make_tensor_descriptor` needs a scratch allocator, so we give it one. Then we pull the compiled
PTX straight out of each kernel's cache and grep for the instructions that only exist on newer
silicon. **Predict before you run:** how many `cp.async.bulk.tensor` will the descriptor kernel emit?


In [ ]:
import re

INSTRUCTIONS = {
    "cp.async.bulk.tensor  (TMA, sm90+)": r"cp\.async\.bulk\.tensor",
    "wgmma                 (sm90)":       r"wgmma",
    "tcgen05               (sm100)":      r"tcgen05",
    "mma.sync              (Ampere TC)":  r"mma\.sync",
    "cp.async              (Ampere)":     r"cp\.async\.(?!bulk)",
    "ldmatrix":                           r"ldmatrix",
}


def ptx_of(kernel):
    """Pull compiled PTX out of a triton JITFunction's device cache."""
    for _dev, entry in kernel.device_caches.items():
        for _key, compiled in entry[0].items():
            if hasattr(compiled, "asm") and "ptx" in compiled.asm:
                return compiled.asm["ptx"]
    return None


if HAS_TRITON:
    def _alloc(size, align, _stream):
        return torch.empty(size, dtype=torch.int8, device="cuda")

    triton.set_allocator(_alloc)

    Z, H, N = 1, 2, 1024
    q, k, v = [torch.randn((Z, H, N, HD), dtype=torch.float16, device="cuda") for _ in range(3)]
    o = torch.empty_like(q)
    Mrow = torch.empty((Z, H, N), device="cuda", dtype=torch.float32)
    grid = (triton.cdiv(N, BM), Z * H, 1)
    strides = (q.stride(0), q.stride(1), q.stride(2), q.stride(3))
    kw = dict(HEAD_DIM=HD, BLOCK_M=BM, BLOCK_N=BN, num_stages=STAGES, num_warps=WARPS)

    outs = {}
    _probe_ptr[grid](q, k, v, 0.5, Mrow, o, *strides, Z, H, N, **kw); outs["plain ptrs"] = o.clone()
    _probe_bp[grid](q, k, v, 0.5, Mrow, o, *strides, Z, H, N, **kw);  outs["block ptrs"] = o.clone()
    _probe_desc[grid](0.5, Mrow, Z, H, q, k, v, o, N, **kw);          outs["descriptors"] = o.clone()

    ref = (torch.softmax((q.float() @ k.float().transpose(2, 3)) * 0.5, dim=-1) @ v.float()).half()
    for name, out in outs.items():
        print(f"  {name:<12} max|err| vs fp32 reference = {(out - ref).abs().max().item():.3e}")

    ptxs = [ptx_of(_probe_desc), ptx_of(_probe_ptr), ptx_of(_probe_bp)]
    cols = ["descriptors", "plain ptrs", "block ptrs"]
    print("\n" + f"{'instruction':<36}" + "".join(f"{c:>13}" for c in cols))
    print("-" * (36 + 13 * len(cols)))
    print(f"{'.target':<36}" + "".join(f"{re.search(r'.target (sm_[0-9a-z]+)', p).group(1):>13}" for p in ptxs))
    for label, pattern in INSTRUCTIONS.items():
        print(f"{label:<36}" + "".join(f"{len(re.findall(pattern, p)):>13}" for p in ptxs))
else:
    print("Recorded on RTX 4080 SUPER (sm_89), pinned config BM=128 BN=64 HD=64 s=2 w=4:\n")
    print("  instruction                            descriptors   plain ptrs   block ptrs")
    print("  .target                                      sm_89        sm_89        sm_89")
    print("  cp.async.bulk.tensor  (TMA, sm90+)               0            0            0")
    print("  wgmma                 (sm90)                     0            0            0")
    print("  tcgen05               (sm100)                    0            0            0")
    print("  mma.sync              (Ampere TC)              128          128          128")
    print("  cp.async              (Ampere)                  22           22           22")
    print("  ldmatrix                                        48           48           48")


### Interpret: the descriptor is *erased*

![All three sources compile to the same sm_89 machine code](../course/figures/fig_t06c_lowering.svg)

Read the zero column. On `sm_89` the descriptor kernel emits **no `cp.async.bulk.tensor`, no
`wgmma`, no `tcgen05`** — and exactly the same 128 `mma.sync` / 48 `ldmatrix` / 22 `cp.async` as
the plain-pointer kernel. The three sources are three spellings of one machine program.

That is the central fact of this notebook:

> **The TMA machinery in the tutorial is not doing anything on your Ampere/Ada GPU.**
> Triton lowers a device tensor descriptor to ordinary `cp.async` + `ldmatrix` loads when the
> target has no TMA unit. You are reading abstraction that the compiler already deleted.

So deleting it from the *source* costs nothing — you are only agreeing with the compiler.

**One wrinkle, and it decides which spelling we adopt.** Run the `_probe_bp` cell on Triton 3.7 and
you get:

```text
UserWarning: tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.
```

Block pointers are on the way out ([Triton
docs](https://triton-lang.org/main/python-api/generated/triton.language.make_block_ptr.html);
[pytorch#154025](https://github.com/pytorch/pytorch/issues/154025)). So the minimal kernel will use
**plain pointer arithmetic** — which is not a nostalgic step backwards, it is exactly how the
tutorial's own `_attn_bwd_dkdv` and `_attn_bwd_dq` have always loaded `Q`, `K`, `V`. The backward
pass never needed a descriptor. Neither does your forward.


## 4. Deletion 3 — `warp_specialize` and the Blackwell accumulator split

**Warp specialization** splits a thread block into producer warps (that fetch tiles) and consumer
warps (that do the MMAs), so the two overlap. The tutorial threads a `warp_specialize` flag all the
way into the inner loop:

```python
for start_n in tl.range(lo, hi, BLOCK_N, warp_specialize=warp_specialize):
```

The Triton documentation for `tl.range` is blunt about where this works:

> *"warp specialization is only supported on Blackwell GPUs and only works on simple matmul loops"*
> — [`triton.language.range`](https://triton-lang.org/main/python-api/generated/triton.language.range.html)

There is a second, subtler branch riding along with it. When `warp_specialize` is on and we are
**not** on Hopper, the tutorial splits the accumulator in half before rescaling it:

```python
if not IS_HOPPER and warp_specialize and BLOCK_M == 128 and HEAD_DIM == 128:
    BM: tl.constexpr = acc.shape[0]
    BN: tl.constexpr = acc.shape[1]
    acc0, acc1 = acc.reshape([BM, 2, BN // 2]).permute(0, 2, 1).split()
    acc0 = acc0 * alpha[:, None]
    acc1 = acc1 * alpha[:, None]
    acc = tl.join(acc0, acc1).permute(0, 2, 1).reshape([BM, BN])
else:
    acc = acc * alpha[:, None]
```

That condition (`not IS_HOPPER`) is true on Ada. So with `warp_specialize=True`, `BLOCK_M=128` and
`HEAD_DIM=128`, **your Ada card takes the split branch** — it just gains nothing from it, because
no warp specialization is happening underneath.


### Is the split branch even doing arithmetic?

`reshape → permute → split → scale each half → join → permute → reshape`. That is a lot of index
gymnastics to multiply by `alpha`. Let's check in NumPy whether it computes anything different from
the one-liner `acc * alpha[:, None]`.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
BM_, BN_ = 128, 128
acc = rng.standard_normal((BM_, BN_)).astype(np.float32)
alpha = rng.random(BM_).astype(np.float32) + 0.5

# the simple path
simple = acc * alpha[:, None]

# the Blackwell/warp-specialize path, transcribed to NumPy.
# tl.split() peels the trailing size-2 axis; tl.join() puts it back.
a = acc.reshape(BM_, 2, BN_ // 2).transpose(0, 2, 1)   # (BM, BN//2, 2)
acc0, acc1 = a[..., 0], a[..., 1]
acc0 = acc0 * alpha[:, None]
acc1 = acc1 * alpha[:, None]
split = np.stack([acc0, acc1], axis=-1).transpose(0, 2, 1).reshape(BM_, BN_)

print("max |split - simple| =", np.abs(split - simple).max())
assert np.array_equal(split, simple), "the split path must be bit-identical"
print("PASS - the accumulator split is an IDENTITY. It changes register layout, not values.")


### Interpret

The split is a **layout hint**, not arithmetic: it exists so the Blackwell backend can keep the two
accumulator halves in separate register banks that its specialized warps consume independently. On
Ada the halves get scaled and immediately rejoined, and the compiler has to prove that dance is a
no-op — costing registers along the way.

Measured on the 4080 SUPER, `warp_specialize=True` versus `False` on the upstream kernel:

| HEAD_DIM | N_CTX | ws=False | ws=True | change |
|---|---|---|---|---|
| 64 | 2048 | 104.9 | 105.1 | +0.2% |
| 64 | 4096 | 106.0 | 106.0 | −0.0% |
| 128 | 2048 | 105.0 | 103.2 | **−1.7%** |
| 128 | 4096 | 106.2 | 104.5 | **−1.5%** |

(TFLOP/s, fp16, BATCH=4, H=32, causal=False.)

At `HEAD_DIM=64` the split branch never fires and the flag is a pure no-op. At `HEAD_DIM=128` it
fires and you *lose* ~1.5%. Note also that the tutorial's own benchmark only ever enables the flag
when `is_blackwell()` or `is_hopper()` — it knows.

**Cost of deleting: zero, and about 1.5% back at `HEAD_DIM=128`.** We drop the flag, the split, and
the `IS_HOPPER` constexpr that steers it.


## 5. Deletion 4 — FP8, the one that actually costs you

Every deletion so far was free. This one is not, and pretending otherwise would be a lie.

FP8 threads through `_attn_fwd` as an `FP8_OUTPUT: tl.constexpr` that picks the dtype, transposes
`V`, and changes the descriptor's block shape:

```python
dtype = tl.float8e5 if FP8_OUTPUT else tl.float16
...
if dtype == tl.float8e5:
    v = desc_v.load([0, offsetv_y]).T   # note: non-transposed V is Blackwell-only
else:
    v = desc_v.load([offsetv_y, 0])
```

It is tempting to file FP8 under "newer-GPU stuff" and delete it. **That would be wrong on Ada.**
Ada's 4th-generation Tensor Cores *do* have FP8 hardware; Ampere's do not
([NVIDIA Ada GPU Architecture whitepaper](https://images.nvidia.com/aem-dam/Solutions/geforce/ada/nvidia-ada-gpu-architecture.pdf)).
Run the tutorial's FP8 path on an RTX 4080 SUPER and it does not merely work — it nearly doubles
throughput.

![FP8 is ~1.9x faster on Ada, and ~1000x less accurate](../course/figures/fig_t06c_fp8.svg)


### So why delete it?

Three reasons, and you should weigh them for *your* card rather than inherit our answer:

| | fp16 | fp8 (e5m2) |
|---|---|---|
| Throughput @ N_CTX=8192 | 107.1 TFLOP/s | **204.1 TFLOP/s** (1.9×) |
| max abs error vs fp32 | 0.0001 | **0.1268** (~1000×) |
| Ampere (sm_80/86) | works | **no hardware** |
| Ada (sm_89) | works | works |
| Non-transposed `V` | works | Blackwell only |

1. **It is not an Ampere/Ada feature, it is an Ada feature.** Our target is both. A kernel that only
   runs on half of its stated targets is not simpler, it is broken in a new way.
2. **e5m2 has 2 mantissa bits.** An error of 0.13 on unit-scale activations is fine for inference
   with a calibrated scale, and not fine as a silent default in a teaching kernel.
3. **It is orthogonal to everything else here.** FP8 is a *numerics* decision; TMA, warp
   specialization and HIP are *portability* scaffolding. Bundling them teaches the wrong lesson.

**Cost of deleting: ~1.9× throughput, on Ada only, if you can absorb the error.** That is the honest
sentence. If you own an Ada or newer card and are serving a quantized model, put FP8 back — but put
it back deliberately, with a scale factor and an accuracy budget, not because it was already there.


## 6. Deletion 5 — the Hopper-only config pruner

The last piece of arch machinery is quiet, and hides in the autotuner:

```python
def keep(conf):
    BLOCK_M = conf.kwargs["BLOCK_M"]
    BLOCK_N = conf.kwargs["BLOCK_N"]
    return not (is_cuda() and torch.cuda.get_device_capability()[0] == 9 and BLOCK_M * BLOCK_N < 128 * 128
                and conf.num_warps == 8)
```

Read the condition: it only ever returns `False` when `capability major == 9`. On Hopper it throws
away small tiles that ask for 8 warps (too few rows per warp to fill an H100 SM). On **anything
else — including your Ada card — `keep` returns `True` for every config it is handed.**

The tutorial builds 36 configs (`BLOCK_M` in {64,128} × `BLOCK_N` in {32,64,128} × `num_stages` in
{2,3,4} × `num_warps` in {4,8}). Let's count how many survive `keep` on each architecture.


In [ ]:
from itertools import product


def upstream_configs() -> list[dict]:
    """The tutorial's 36-config grid."""
    return [{"BLOCK_M": bm, "BLOCK_N": bn, "num_stages": s, "num_warps": w}
            for bm, bn, s, w in product([64, 128], [32, 64, 128], [2, 3, 4], [4, 8])]


def keep(conf: dict, backend: str, major: int) -> bool:
    """Verbatim port of the tutorial's keep()."""
    is_cuda = backend == "cuda"
    return not (is_cuda and major == 9
                and conf["BLOCK_M"] * conf["BLOCK_N"] < 128 * 128
                and conf["num_warps"] == 8)


all_cfgs = upstream_configs()
print(f"configs built by the tutorial: {len(all_cfgs)}\n")
for label, backend, major in ARCHS:
    kept = sum(keep(c, backend, major) for c in all_cfgs)
    pruned = len(all_cfgs) - kept
    print(f"  {label:<16} keeps {kept:>2} / {len(all_cfgs)}   (prunes {pruned})")

assert sum(keep(c, "cuda", 8) for c in all_cfgs) == 36, "Ada prunes nothing"
assert sum(keep(c, "cuda", 9) for c in all_cfgs) == 21, "Hopper prunes 15"
print("\nOn Ampere/Ada keep() prunes 0 of 36 configs. It is an identity filter.")


### Interpret

`keep()` is a **no-op on our target**. We delete it — and while we are here we also shrink the grid
itself. The tutorial's `_attn_fwd` carries `tl.static_assert(BLOCK_N <= HEAD_DIM)`, so at
`HEAD_DIM=64` the `BLOCK_N=128` configs can never compile anyway. Dropping `BLOCK_N=128` takes the
grid from 36 configs to **24**, and the autotuner stops paying to compile twelve variants it will
throw away.

What we *keep* is `prune_invalid_configs`, because that one is not about architecture at all — it
guards **shape** (`BLOCK_M <= N_CTX`, and `BLOCK_M >= BLOCK_N` when causal). Deleting a gate is only
safe once you have read what it guards.


## 7. The kernel that remains

Five deletions later, here is the whole forward pass. It is the same FlashAttention-2 you met in
06a — online softmax, two causal stages, one `exp2`-based rescale — with nothing between you and
the loads.

![What the strip removes](../course/figures/fig_t06c_strip.svg)

Two things to notice as you read:

- **`K_ptr` is built transposed.** `offs_n[None, :] * stride_kn + offs_d[:, None] * stride_kk` yields
  a `(HEAD_DIM, BLOCK_N)` tile, so `tl.dot(q, k)` needs no explicit `.T`. This is precisely the trick
  `_attn_bwd_dkdv` uses for `qT_ptrs`.
- **`STAGE` is untouched.** `stage = 3 if causal else 1`; the kernel calls the inner loop with
  `4 - STAGE` (off-band) and then `2` (the masked diagonal). None of that was ever arch-specific.


In [ ]:
configs = [
    triton.Config({"BLOCK_M": BLOCK_M, "BLOCK_N": BLOCK_N}, num_stages=s, num_warps=w)
    for BLOCK_M in [64, 128]
    for BLOCK_N in [32, 64]
    for s in [2, 3, 4]
    for w in [4, 8]
]


def prune_invalid_configs(configs, named_args, **kwargs):
    """SHAPE guard, not an arch guard: BLOCK_M must fit N_CTX, and cover BLOCK_N when causal."""
    N_CTX, STAGE = kwargs["N_CTX"], kwargs["STAGE"]
    return [c for c in configs
            if c.kwargs["BLOCK_M"] <= N_CTX
            and (c.kwargs["BLOCK_M"] >= c.kwargs["BLOCK_N"] or STAGE == 1)]


@triton.jit
def _fwd_inner(acc, l_i, m_i, q, K_ptr, V_ptr, stride_kn, stride_vn, start_m, qk_scale,
               BLOCK_M: tl.constexpr, HEAD_DIM: tl.constexpr, BLOCK_N: tl.constexpr,
               STAGE: tl.constexpr, offs_m: tl.constexpr, offs_n: tl.constexpr, N_CTX: tl.constexpr):
    if STAGE == 1:                                  # off-band: strictly below the diagonal
        lo, hi = 0, start_m * BLOCK_M
    elif STAGE == 2:                                # on-band: the masked diagonal block
        lo, hi = start_m * BLOCK_M, (start_m + 1) * BLOCK_M
        lo = tl.multiple_of(lo, BLOCK_M)
    else:                                           # non-causal: every key block
        lo, hi = 0, N_CTX

    K_ptr += lo * stride_kn
    V_ptr += lo * stride_vn

    for start_n in tl.range(lo, hi, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        k = tl.load(K_ptr)                          # (HEAD_DIM, BLOCK_N), pre-transposed
        qk = tl.dot(q, k)
        if STAGE == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)
            m_ij = tl.maximum(m_i, tl.max(qk, 1))
            qk -= m_ij[:, None]
        else:
            m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
            qk = qk * qk_scale - m_ij[:, None]
        p = tl.math.exp2(qk)
        alpha = tl.math.exp2(m_i - m_ij)            # rescale the old accumulator
        l_ij = tl.sum(p, 1)
        acc = acc * alpha[:, None]                  # (no Blackwell split - see section 4)
        v = tl.load(V_ptr)                          # (BLOCK_N, HEAD_DIM)
        acc = tl.dot(p.to(tl.float16), v, acc)
        l_i = l_i * alpha + l_ij
        m_i = m_ij
        K_ptr += BLOCK_N * stride_kn
        V_ptr += BLOCK_N * stride_vn
    return acc, l_i, m_i


In [ ]:
@triton.autotune(configs=configs, key=["N_CTX", "HEAD_DIM"],
                 prune_configs_by={"early_config_prune": prune_invalid_configs})
@triton.jit
def _attn_fwd(Q, K, V, sm_scale, M, Out,
              stride_qz, stride_qh, stride_qm, stride_qk,
              stride_kz, stride_kh, stride_kn, stride_kk,
              stride_vz, stride_vh, stride_vn, stride_vk,
              stride_oz, stride_oh, stride_om, stride_on,
              Z, H, N_CTX,
              HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr,
              BLOCK_N: tl.constexpr, STAGE: tl.constexpr):
    tl.static_assert(BLOCK_N <= HEAD_DIM)
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z, off_h = off_hz // H, off_hz % H

    q_base = Q + off_z.to(tl.int64) * stride_qz + off_h.to(tl.int64) * stride_qh
    k_base = K + off_z.to(tl.int64) * stride_kz + off_h.to(tl.int64) * stride_kh
    v_base = V + off_z.to(tl.int64) * stride_vz + off_h.to(tl.int64) * stride_vh
    o_base = Out + off_z.to(tl.int64) * stride_oz + off_h.to(tl.int64) * stride_oh

    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, HEAD_DIM)

    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32) + 1.0
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)
    qk_scale = sm_scale * 1.44269504                # fold 1/log(2) so we can use exp2

    q = tl.load(q_base + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk)
    K_ptr = k_base + offs_n[None, :] * stride_kn + offs_d[:, None] * stride_kk
    V_ptr = v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vk

    if STAGE & 1:                                   # causal -> off-band; non-causal -> everything
        acc, l_i, m_i = _fwd_inner(acc, l_i, m_i, q, K_ptr, V_ptr, stride_kn, stride_vn,
                                   start_m, qk_scale, BLOCK_M, HEAD_DIM, BLOCK_N,
                                   4 - STAGE, offs_m, offs_n, N_CTX)
    if STAGE & 2:                                   # causal only -> the masked diagonal
        acc, l_i, m_i = _fwd_inner(acc, l_i, m_i, q, K_ptr, V_ptr, stride_kn, stride_vn,
                                   start_m, qk_scale, BLOCK_M, HEAD_DIM, BLOCK_N,
                                   2, offs_m, offs_n, N_CTX)

    m_i += tl.math.log2(l_i)                        # logsumexp, for the backward pass
    acc = acc / l_i[:, None]
    tl.store(M + off_hz * N_CTX + offs_m, m_i)
    tl.store(o_base + offs_m[:, None] * stride_om + offs_d[None, :] * stride_on, acc.to(tl.float16))


def attention_minimal(q, k, v, causal: bool, sm_scale: float):
    """Launch the stripped forward. fp16, CUDA, Ampere/Ada. No TMA, no FP8, no warp specialization."""
    HEAD_DIM = q.shape[-1]
    assert HEAD_DIM in {16, 32, 64, 128, 256}
    assert q.shape[2] % 128 == 0, "no boundary masking: N_CTX must be a multiple of BLOCK_M (Exercise E)"
    o = torch.empty_like(q)
    M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
    stage = 3 if causal else 1
    grid = lambda META: (triton.cdiv(q.shape[2], META["BLOCK_M"]), q.shape[0] * q.shape[1], 1)
    _attn_fwd[grid](
        q, k, v, sm_scale, M, o,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        o.stride(0), o.stride(1), o.stride(2), o.stride(3),
        q.shape[0], q.shape[1], N_CTX=q.shape[2],
        HEAD_DIM=HEAD_DIM, STAGE=stage)
    return o, M

print("minimal kernel defined:", len(configs), "autotune configs (upstream had 36)")


## 8. Does it still compute attention?

Deleting code that "doesn't do anything" is exactly how kernels acquire silent numerical bugs. The
only acceptable evidence is a comparison against a reference. We sweep batch, heads, sequence
length, head dim, and both causal modes, against a plain PyTorch softmax-attention in fp32.


In [ ]:
def reference_attention(q, k, v, causal: bool, sm_scale: float):
    """Textbook attention in fp32, the ground truth."""
    N = q.shape[2]
    p = torch.matmul(q.float(), k.float().transpose(2, 3)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones((N, N), device=q.device))
        p[:, :, mask == 0] = float("-inf")
    p = torch.softmax(p, dim=-1)
    return torch.matmul(p, v.float()).half()


if HAS_TRITON:
    torch.manual_seed(20)
    shapes = [(1, 2, 1024, 64), (1, 2, 1024, 128), (4, 8, 2048, 64)]
    print(f"{'Z':>2} {'H':>3} {'N_CTX':>6} {'HEAD_DIM':>9} {'causal':>7} {'max|err|':>11}   status")
    worst = 0.0
    for (Z_, H_, N_, D_) in shapes:
        for causal in (False, True):
            qq, kk, vv = [torch.empty((Z_, H_, N_, D_), dtype=torch.float16, device="cuda")
                          .normal_(mean=0.0, std=0.5) for _ in range(3)]
            out, _ = attention_minimal(qq, kk, vv, causal, 0.5)
            err = (out - reference_attention(qq, kk, vv, causal, 0.5)).abs().max().item()
            worst = max(worst, err)
            print(f"{Z_:>2} {H_:>3} {N_:>6} {D_:>9} {str(causal):>7} {err:>11.3e}   "
                  f"{'OK' if err < 1e-2 else 'FAIL'}")
            del qq, kk, vv
            torch.cuda.empty_cache()
    assert worst < 1e-2, f"minimal kernel diverged from the reference: {worst}"
    print(f"\nPASS - worst error {worst:.3e} over all shapes (fp16 tolerance is 1e-2).")
else:
    print("Recorded on RTX 4080 SUPER: worst max|err| = 9.766e-04 across all six cases (tol 1e-2).")


## 9. Does it still fly?

Correctness is half the claim. The other half — the one that justifies the deletions — is that the
stripped kernel is not slower. We benchmark it against
`torch.nn.functional.scaled_dot_product_attention`, which dispatches to PyTorch's own hand-tuned
Flash kernel, at the tutorial's benchmark shape: `BATCH=4, H=32, HEAD_DIM=64`, fp16.

TFLOP/s counts `2 · 2 · BATCH · H · N_CTX² · HEAD_DIM` (two matmuls), halved when causal.


In [ ]:
if HAS_TRITON:
    import torch.nn.functional as F

    BATCH, NH, HEAD_DIM = 4, 32, 64

    def tflops(ms: float, N: int, causal: bool) -> float:
        flops = 2.0 * 2.0 * BATCH * NH * N * N * HEAD_DIM
        if causal:
            flops *= 0.5
        return flops * 1e-12 / (ms * 1e-3)

    for causal in (False, True):
        print(f"\ncausal={causal}   (TFLOP/s, higher is better)")
        print(f"{'N_CTX':>7} {'minimal':>10} {'torch SDPA':>12} {'ratio':>8}")
        for N in [1024, 2048, 4096]:
            qq, kk, vv = [torch.randn((BATCH, NH, N, HEAD_DIM), dtype=torch.float16, device="cuda")
                          for _ in range(3)]
            ms_min = triton.testing.do_bench(lambda: attention_minimal(qq, kk, vv, causal, 1.3))
            ms_sdpa = triton.testing.do_bench(
                lambda: F.scaled_dot_product_attention(qq, kk, vv, is_causal=causal, scale=1.3))
            a, b = tflops(ms_min, N, causal), tflops(ms_sdpa, N, causal)
            print(f"{N:>7} {a:>10.1f} {b:>12.1f} {a / b:>7.2f}x")
            del qq, kk, vv
            torch.cuda.empty_cache()
else:
    print("Recorded on RTX 4080 SUPER (sm_89), fp16, BATCH=4 H=32 HEAD_DIM=64 - see the figure below.")
    print("At N_CTX=8192, causal=False:  upstream 107.1 | minimal 106.0 | torch SDPA 104.2 TFLOP/s")


### Interpret: you lost about one percent

![Minimal keeps up with upstream on Ada](../course/figures/fig_t06c_bench.svg)

Recorded on the reference machine (TFLOP/s, fp16, `BATCH=4, H=32, HEAD_DIM=64`):

| | N_CTX=1024 | 2048 | 4096 | 8192 |
|---|---|---|---|---|
| **causal=False** | | | | |
| upstream (tutorial) | 101.1 | 104.6 | 105.8 | 107.1 |
| minimal (plain ptrs) | 100.1 | 103.3 | 104.4 | 106.0 |
| torch SDPA | 98.0 | 101.7 | 103.0 | 104.2 |
| **causal=True** | | | | |
| upstream (tutorial) | 71.7 | 85.9 | 93.9 | 97.9 |
| minimal (plain ptrs) | 72.7 | 84.7 | 93.8 | 95.2 |
| minimal (block ptrs) | 77.7 | 91.9 | 99.7 | **103.7** |
| torch SDPA | 73.6 | 89.0 | 96.6 | 100.6 |

Half the source, within **1–3%** of upstream's throughput. Non-causal, the stripped kernel edges out
`torch.nn.functional`'s own Flash kernel at every length; causal, it trails it by 1–5%. **Nothing you
deleted from the fp16 path was making your GPU faster.** (FP8, from §5, is the exception that proves
it — and that one you deleted knowingly.)

There is one honest anomaly, and it is worth sitting with. In the causal case the **deprecated**
block-pointer variant is the fastest thing on the chart — up to **+6% over upstream** at
`N_CTX=8192`. A block pointer carries shape, strides and an `order=(1,0)` contiguity promise into
the IR; plain pointer arithmetic makes the compiler re-derive that from raw integer math, and in the
causal inner loop it does not always succeed. So the deprecated API wins *today*, on *this* kernel,
on *this* card — a good reminder that "minimal" and "fastest" are different objectives, and that a
deprecation is a statement about where an API is going, not about where the performance is.


## 10. What you gave up, and where you would put it back

Deleting code is only responsible if you can say what the code was for. Here is the ledger.

| Deleted | Cost on Ampere/Ada | Put it back when |
|---|---|---|
| `is_hip()` branch, `waves_per_eu` | none | you target AMD CDNA |
| host `TensorDescriptor` + `_host_descriptor_pre_hook` | none (guarded by cap ≥ 9) | you target Hopper/Blackwell |
| device `tl.make_tensor_descriptor` | none — compiles to the same PTX (§3) | you target Hopper/Blackwell |
| `warp_specialize` + `tl.range(warp_specialize=…)` | none — you *gain* ~1.5% at HEAD_DIM=128 | you target Blackwell |
| Blackwell accumulator split | none — it is an identity (§4) | you target Blackwell |
| `maxnreg` kernel arg | none | you target Blackwell |
| `keep()` config pruner | none — prunes 0 of 36 (§6) | you target Hopper |
| `BLOCK_N=128` configs | none at HEAD_DIM ≤ 64 (`static_assert`) | HEAD_DIM ≥ 128 |
| **FP8 (`float8e5`) path** | **~1.9× throughput, on Ada only** | you serve a quantized model *and* have Ada+ |
| descriptors' implicit bounds checks | **N_CTX must divide BLOCK_M** | your sequences are ragged (Exercise E) |

Read the last two rows again. Those are the real trades. Everything above them was scaffolding for
hardware you do not own.


> **Going to production.** This kernel is a *teaching* artifact: fp16, forward-only, and it assumes
> `N_CTX % BLOCK_M == 0`. To take it further:
>
> - **Backward pass:** reuse the tutorial's, unmodified. `_attn_bwd_preprocess`, `_attn_bwd_dkdv`,
>   `_attn_bwd_dq` and `_attn_bwd` contain **zero** arch gates — they already load with plain
>   pointers. That is not a coincidence; it is the point of §3. Wire both into a
>   `torch.autograd.Function` exactly as 06b §10 shows.
> - **Ragged sequences:** add `mask=`/`other=0.0` to each `tl.load` and mask the tail of `qk`
>   (Exercise E), or pad up to a multiple of `BLOCK_M`. Descriptors gave you this for free;
>   pointers do not.
> - **Real deployment:** you almost certainly want
>   [FlashAttention](https://github.com/Dao-AILab/flash-attention) or
>   `torch.nn.functional.scaled_dot_product_attention`, which carry the Hopper/Blackwell paths you
>   just deleted, plus dropout, ALiBi, GQA and variable-length batching. Build this kernel to
>   *understand* those; ship theirs.
> - **Multiple architectures in one codebase:** don't fork the kernel. Keep the arch gates — but now
>   you know what each one is worth, and you can read them as the compile-time switches they are
>   rather than as mysteries.


## Exercises

Five, in rising order of difficulty. All of them are **pure NumPy/Python** — they run whether or not
you have a GPU. Each has a collapsible solution; try it before you peek.


### Exercise A — which paths does a given GPU take?

Write `enabled_paths(backend, major)` returning the **set** of code paths a GPU activates in the
tutorial. The five path names are `"hip_tuning"`, `"host_descriptor"`, `"hopper_pruning"`,
`"blackwell_maxnreg"`, `"warp_specialization"`.

Rules, straight from the tutorial: HIP tuning on AMD; host descriptors at capability ≥ 9; Hopper
pruning at exactly 9; `maxnreg` and real warp specialization at exactly 10.

**Predict first:** how many paths does an Ada card enable?


In [ ]:
def enabled_paths(backend: str, major: int) -> set[str]:
    raise NotImplementedError("your turn")


assert enabled_paths("hip", 9) == {"hip_tuning"}
assert enabled_paths("cuda", 8) == set()                      # Ampere AND Ada: nothing
assert enabled_paths("cuda", 9) == {"host_descriptor", "hopper_pruning"}
assert enabled_paths("cuda", 10) == {"host_descriptor", "blackwell_maxnreg", "warp_specialization"}
print("PASS - an Ampere/Ada card enables ZERO of the five special paths.")


<details>
<summary>▶ Show solution</summary>

```python
def enabled_paths(backend: str, major: int) -> set[str]:
    is_cuda = backend == "cuda"
    paths = set()
    if backend == "hip":
        paths.add("hip_tuning")
    if is_cuda and major >= 9:
        paths.add("host_descriptor")
    if is_cuda and major == 9:
        paths.add("hopper_pruning")
    if is_cuda and major == 10:
        paths.add("blackwell_maxnreg")
        paths.add("warp_specialization")
    return paths
```

</details>

### Exercise B — how much does `keep()` actually prune?

You saw the answer for Ada (nothing). Now compute it for Hopper *from the rule*, without running
Triton: of the 36 configs, how many does `keep()` drop when `major == 9`?

Write `pruned_on_hopper()` returning that count. Remember the rule drops a config only when
`BLOCK_M * BLOCK_N < 128*128` **and** `num_warps == 8`.

**Predict first:** the grid has 36 configs and 6 distinct `(BLOCK_M, BLOCK_N)` pairs.


In [ ]:
def pruned_on_hopper() -> int:
    raise NotImplementedError("your turn")


n = pruned_on_hopper()
print(f"Hopper prunes {n} of 36 configs, keeping {36 - n}.")
assert n == 15, f"expected 15, got {n}"

# and the tile pairs that survive with 8 warps:
survivors = [(bm, bn) for bm in (64, 128) for bn in (32, 64, 128) if bm * bn >= 128 * 128]
assert survivors == [(128, 128)], survivors
print("Only the (128,128) tile is big enough to justify 8 warps on an H100 SM.")
print("PASS")


<details>
<summary>▶ Show solution</summary>

```python
from itertools import product


def pruned_on_hopper() -> int:
    dropped = 0
    for bm, bn, _s, w in product([64, 128], [32, 64, 128], [2, 3, 4], [4, 8]):
        if bm * bn < 128 * 128 and w == 8:
            dropped += 1
    return dropped

# 5 of the 6 (BLOCK_M, BLOCK_N) pairs are under 16384 elements;
# each appears with 3 num_stages values, and only for num_warps == 8:
#   5 pairs x 3 stages x 1 warp-count = 15
```

</details>

### Exercise C — the causal `STAGE` ranges

Reconstruct the inner loop's bounds. Write `stage_range(STAGE, start_m, BLOCK_M, N_CTX)` returning
`(lo, hi)` exactly as `_fwd_inner` computes it, then use the provided `n_steps(...)` to count how
many `BLOCK_N`-sized iterations that range costs.

Recall: outer `STAGE=3` (causal) calls the inner loop twice — once with `4 - 3 = 1`, once with `2`.
Outer `STAGE=1` (non-causal) calls it once with `4 - 1 = 3`.

**Predict first:** at `N_CTX=1024, BLOCK_M=128, BLOCK_N=64`, does program `start_m=3` do more work
causal or non-causal?


In [ ]:
def stage_range(STAGE: int, start_m: int, BLOCK_M: int, N_CTX: int) -> tuple[int, int]:
    raise NotImplementedError("your turn")


def n_steps(STAGE: int, start_m: int, BLOCK_M: int, BLOCK_N: int, N_CTX: int) -> int:
    lo, hi = stage_range(STAGE, start_m, BLOCK_M, N_CTX)
    return (hi - lo) // BLOCK_N


N_CTX_, BLOCK_M_, BLOCK_N_, start_m_ = 1024, 128, 64, 3
assert stage_range(1, start_m_, BLOCK_M_, N_CTX_) == (0, 384)      # off-band: keys 0..start_m*BM
assert stage_range(2, start_m_, BLOCK_M_, N_CTX_) == (384, 512)    # the diagonal block
assert stage_range(3, start_m_, BLOCK_M_, N_CTX_) == (0, 1024)     # everything

causal = (n_steps(1, start_m_, BLOCK_M_, BLOCK_N_, N_CTX_)
          + n_steps(2, start_m_, BLOCK_M_, BLOCK_N_, N_CTX_))
non_causal = n_steps(3, start_m_, BLOCK_M_, BLOCK_N_, N_CTX_)
print(f"program start_m={start_m_}: causal does {causal} steps, non-causal does {non_causal}")
assert (causal, non_causal) == (8, 16)
print("PASS - causal work grows with start_m; program 0 does almost nothing, the last does it all.")


<details>
<summary>▶ Show solution</summary>

```python
def stage_range(STAGE: int, start_m: int, BLOCK_M: int, N_CTX: int) -> tuple[int, int]:
    if STAGE == 1:                                   # off-band, strictly below the diagonal
        return 0, start_m * BLOCK_M
    if STAGE == 2:                                   # on-band, the masked diagonal block
        return start_m * BLOCK_M, (start_m + 1) * BLOCK_M
    return 0, N_CTX                                  # STAGE == 3: non-causal, all key blocks

# This is also why causal attention load-balances badly: program start_m=0 runs 2 steps,
# program start_m=7 runs 16. FlashAttention-2 accepts that; later work re-partitions it.
```

</details>

### Exercise D — prove the Blackwell split is free, in general

§4 checked the accumulator split for one random matrix at `BLOCK_M = BLOCK_N = 128`. Generalise it.
Write `blackwell_split_scale(acc, alpha)` that reproduces the tutorial's
`reshape → permute → split → scale → join → permute → reshape`, and show it is **bit-identical** to
`acc * alpha[:, None]` for several shapes.

Hint: `tl.split()` peels a trailing axis of size 2; `tl.join()` stacks two tensors into one.

**Predict first:** could this ever *not* be an identity — say, for odd `BLOCK_N`?


In [ ]:
def blackwell_split_scale(acc: np.ndarray, alpha: np.ndarray) -> np.ndarray:
    raise NotImplementedError("your turn")


rng2 = np.random.default_rng(7)
for bm, bn in [(64, 64), (128, 128), (32, 256)]:
    a_ = rng2.standard_normal((bm, bn)).astype(np.float32)
    al_ = (rng2.random(bm) + 0.5).astype(np.float32)
    got, want = blackwell_split_scale(a_, al_), a_ * al_[:, None]
    assert np.array_equal(got, want), f"mismatch at {(bm, bn)}: {np.abs(got - want).max()}"
print("PASS - bit-identical at every shape. The split moves registers, not numbers.")
print("(BN must be even for the reshape to (BM, 2, BN//2) to exist - which is why the")
print(" tutorial guards it with `BLOCK_M == 128 and HEAD_DIM == 128`.)")


<details>
<summary>▶ Show solution</summary>

```python
def blackwell_split_scale(acc: np.ndarray, alpha: np.ndarray) -> np.ndarray:
    BM, BN = acc.shape
    # acc.reshape([BM, 2, BN // 2]).permute(0, 2, 1)  ->  (BM, BN//2, 2)
    a = acc.reshape(BM, 2, BN // 2).transpose(0, 2, 1)
    acc0, acc1 = a[..., 0], a[..., 1]                 # tl.split()
    acc0 = acc0 * alpha[:, None]
    acc1 = acc1 * alpha[:, None]
    joined = np.stack([acc0, acc1], axis=-1)          # tl.join()  -> (BM, BN//2, 2)
    return joined.transpose(0, 2, 1).reshape(BM, BN)

# Scaling by alpha[:, None] is applied per ROW. Both halves are column-slices of the same
# rows, so scaling them separately and re-interleaving cannot change any value.
```

</details>

### Exercise E — pay for what the descriptors gave you

Descriptors bounds-check for free: a TMA (or emulated) load past the end of the tensor returns
zeros. Plain pointers do not — which is why `attention_minimal` asserts `N_CTX % 128 == 0`.

Emulate a masked load. Write `masked_load_kt(K, start_n, BLOCK_N, N_CTX)` that returns the
**transposed** key tile of shape `(HEAD_DIM, BLOCK_N)` for rows `[start_n, start_n + BLOCK_N)`,
filling any row at or beyond `N_CTX` with `0.0` — the NumPy twin of
`tl.load(K_ptr, mask=offs_n[None, :] < N_CTX, other=0.0)`.

**Predict first:** with `N_CTX=1000` and `BLOCK_N=64`, how many columns of the last tile are real?


In [ ]:
def masked_load_kt(K: np.ndarray, start_n: int, BLOCK_N: int, N_CTX: int) -> np.ndarray:
    raise NotImplementedError("your turn")


HEAD_DIM_E, N_CTX_E, BLOCK_N_E = 8, 1000, 64
K_e = np.arange(N_CTX_E * HEAD_DIM_E, dtype=np.float32).reshape(N_CTX_E, HEAD_DIM_E)

full = masked_load_kt(K_e, 0, BLOCK_N_E, N_CTX_E)
assert full.shape == (HEAD_DIM_E, BLOCK_N_E)
assert np.array_equal(full, K_e[0:64].T), "an interior tile must be an exact transpose"

last_start = (N_CTX_E // BLOCK_N_E) * BLOCK_N_E        # 960
tail = masked_load_kt(K_e, last_start, BLOCK_N_E, N_CTX_E)
n_real = N_CTX_E - last_start                          # 40
assert tail.shape == (HEAD_DIM_E, BLOCK_N_E)
assert np.array_equal(tail[:, :n_real], K_e[last_start:N_CTX_E].T)
assert np.all(tail[:, n_real:] == 0.0), "out-of-range columns must be zero, not garbage"
print(f"last tile: {n_real} real columns, {BLOCK_N_E - n_real} zero-filled. PASS")
print("Zeros are safe for the VALUES; the SCORES still need a mask - see the solution.")


<details>
<summary>▶ Show solution</summary>

```python
def masked_load_kt(K: np.ndarray, start_n: int, BLOCK_N: int, N_CTX: int) -> np.ndarray:
    HEAD_DIM = K.shape[1]
    offs_n = start_n + np.arange(BLOCK_N)              # the rows this tile wants
    mask = offs_n < N_CTX                              # tl.load(..., mask=..., other=0.0)
    tile = np.zeros((BLOCK_N, HEAD_DIM), dtype=K.dtype)
    tile[mask] = K[offs_n[mask]]
    return tile.T                                      # (HEAD_DIM, BLOCK_N)

# In the real kernel you must also mask the SCORES, not just the loads: a zero-filled key
# column still produces qk = 0, and exp2(0) = 1, which would pollute l_i and the softmax
# denominator. The tutorial does exactly this for the causal diagonal with
#     qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)
# A ragged tail needs the same trick, with mask = (start_n + offs_n) < N_CTX.
```

</details>

## Further Reading

**Source of truth**

- The tutorial this notebook strips: [`06-fused-attention.py`](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)
  (OpenAI Triton kernel team) — and its local copy, `tutorials_jupyter/06-fused-attention.ipynb`.
- [`triton.language.range`](https://triton-lang.org/main/python-api/generated/triton.language.range.html)
  — *"warp specialization is only supported on Blackwell GPUs and only works on simple matmul loops."*
- [`triton.language.make_tensor_descriptor`](https://triton-lang.org/main/python-api/generated/triton.language.make_tensor_descriptor.html)
  — *"On NVIDIA GPUs with TMA support, this will result in a TMA descriptor object…"*
- [`triton.language.make_block_ptr`](https://triton-lang.org/main/python-api/generated/triton.language.make_block_ptr.html)
  — deprecated; see [pytorch#154025](https://github.com/pytorch/pytorch/issues/154025) for the migration.
- [CUDA Programming Guide — Asynchronous Data Copies](https://docs.nvidia.com/cuda/cuda-programming-guide/04-special-topics/async-copies.html)
  — TMA is *"compute capability 9.0 (Hopper) and later"*.
- [NVIDIA Ada GPU Architecture whitepaper](https://images.nvidia.com/aem-dam/Solutions/geforce/ada/nvidia-ada-gpu-architecture.pdf)
  — 4th-gen Tensor Cores add FP8; Ampere has none.


**Going deeper**

- [Warp Specialization in Triton: Design and Roadmap](https://pytorch.org/blog/warp-specialization-in-triton-design-and-roadmap/) (PyTorch blog).
- [OpenAI Triton on NVIDIA Blackwell](https://developer.nvidia.com/blog/openai-triton-on-nvidia-blackwell-boosts-ai-performance-and-programmability/) (NVIDIA blog).
- [CUTLASS Tutorial: Mastering the NVIDIA Tensor Memory Accelerator](https://research.colfax-intl.com/tutorial-hopper-tma/) (Colfax Research) — what you would gain on an H100.
- [Deep Dive on the Hopper TMA Unit for FP8 GEMMs](https://pytorch.org/blog/hopper-tma-unit/) (PyTorch blog).
- FlashAttention-2: [arXiv:2307.08691](https://arxiv.org/abs/2307.08691); the original FlashAttention: [arXiv:2205.14135](https://arxiv.org/abs/2205.14135).

**In this course**

- **06a** — the forward pass, unpacked (online softmax, causal `STAGE`).
- **06b** — the backward pass, unpacked (`D = rowsum(O∘dO)`, recompute-from-`M`, the two-axis split).
- **Chapter 16b** — the same fused attention written in raw CUDA C++.
- **Chapters 16c / 16d** — FlashAttention-3 and 4, which are largely *about* the Hopper/Blackwell
  machinery you just deleted.


## Recap — portability is a feature you pay for

You took a 760-line multi-architecture kernel, deleted everything an Ampere/Ada card cannot execute,
and ended up with a ~120-line forward pass that agrees with an fp32 reference to **9.8e-4** and runs
**within a few percent** of the original.

| What you did | Where | The evidence |
|---|---|---|
| Named the five arch gates | §0 | Ada trips exactly one: `is_cuda` |
| Deleted the HIP branch | §2 | dead on any NVIDIA card |
| Deleted TMA descriptors | §3 | **identical PTX** — 0 `cp.async.bulk.tensor` on sm_89 |
| Deleted `warp_specialize` + the acc split | §4 | the split is a bit-exact identity; the flag costs 1.5% at HEAD_DIM=128 |
| Kept FP8 out, knowingly | §5 | ~1.9× on Ada, ~1000× the error, absent on Ampere |
| Deleted the Hopper config pruner | §6 | prunes 0 of 36 configs on Ada |
| Rebuilt and verified the kernel | §7–§9 | worst error 9.8e-4; 106 vs 107 TFLOP/s |


**The transferable lesson.** A published kernel is a *union* of the kernels its authors had to ship.
Reading it as if every line runs on your GPU will confuse you; the `if`s are compile-time switches,
and on your card most of them are already `False`. Before you optimize a kernel you did not write:
**find its gates, evaluate them for your hardware, and delete the branches you cannot take.** What
remains is the kernel you are actually running — and it is usually small enough to hold in your head.

Then, before you believe any of it, do what §3 and §8 did: **read the PTX, and check against a
reference.** "It compiles to the same thing" is a claim about a compiler, and compilers change.

**Where to go next.** **Chapters 16c/16d** take the opposite journey — they add the Hopper and
Blackwell machinery back, deliberately, and show what FlashAttention-3 and -4 buy with it. Having
deleted it once, you will recognise every piece.
